Nicht vergesessen, dass in calc_wav_coh_bvpa_pr noch die ch_list richtig aktiviert gehört.

interpft nochmal überprüfen, ob die Änderungen (3 mal eingerückt) auch passen (Stw. Downsampling).

Handling von negativen Weren in amp

### @User: Choose measurement, and channel for plots

In [1]:
measurement = '01'
channel_for_plot = 'S1D15'

### Initializing

In [ ]:
### Imports
%matplotlib qt

import importlib
import cedalion
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import xarray as xr
from cedalion.io.snirf import read_snirf
from cedalion.nirs.cw import int2od
from cedalion.nirs.cw import od2conc

pd.set_option('display.max_rows', 10)
xr.set_options(display_expand_data=False)

def calc_concentratoin(rec):
    od = int2od(rec["amp"])
    dpf = xr.DataArray([6, 6], dims="wavelength", coords={"wavelength" : od.wavelength})
    return od2conc(od, rec.geo3d, dpf)


In [3]:
### Load recording, clalculate concentration
days =     [ '03',  '05',  '05',  '05',  '06',  '06',  '06',  '06',  '07',  '07',
             '08',  '08',  '08',  '08',  '09',  '09',  '10',  '10',  '11',  '13',
             '13',  '13',  '13',  '13',  '14',  '14']
sessions = ['001', '001', '002', '003', '001', '002', '003', '004', '001', '002',
            '001', '002', '003', '004', '001', '002', '001', '002', '001', '001',
            '002', '003', '004', '005', '001', '002']

m = measurement
d = days[int(m)-1]
s = sessions[int(m)-1]

path_to_snirf_file = (
    f"D:/WorkPy/04_data_mainstudy/Data/measurement_{m}/2025-05-{d}_{s}/2025-05-{d}_{s}_inclAUG.snirf")

recordings = read_snirf(path_to_snirf_file)
rec = recordings[0]
# rec["amp"] = rec["amp"].where(rec["amp"] > 0, np.nan)
rec["conc"] = calc_concentratoin(rec)

In [ ]:
# show xarray of concentation time series
rec['conc']

### Creation of Bvp_Container --> BVP extraction --> Waveform: extraction

In [ ]:
# for dev
importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)
importlib.reload(cedalion.dataclasses.bvp_container)

In [4]:
### create Bvp_Container
from cedalion.dataclasses.bvp_container import BVP_Container
bvp_cont = BVP_Container()

In [ ]:
# show Bvp_Container
bvp_cont

In [5]:
### extract bvp
from cedalion.sigproc.bvp_wav_ana_v12 import extract_bvp
bvp_ts = extract_bvp(rec["conc"].sel(chromo="HbO"))
bvp_cont['bvp_ts'] = bvp_ts

In [ ]:
# show updated BVP_Container and xarray of BVP time series
print(bvp_cont)
display(bvp_cont['bvp_ts'])

In [6]:
### extract waveforms
from cedalion.sigproc.bvp_wav_ana_v12 import extract_waveforms
wav_storage_user, wav_storage_details = extract_waveforms(bvp_ts.sel(compound="bvp_ts"))
bvp_cont.wav_storage_user = wav_storage_user
bvp_cont.wav_storage_details = wav_storage_details

In [ ]:
# show updated BVP_Container
print(bvp_cont)

Plots

In [7]:
channel_for_plot = rec['amp'].channel.values[0]

In [8]:
# plot waveforms
from cedalion.sigproc.bvp_wav_ana_v12 import plot_wavs_4x
plot_wavs_4x(bvp_cont, channel_for_plot)

In [9]:
# plot [HbO] and bvp time series
from cedalion.sigproc.bvp_wav_ana_v12 import plot_concts_bvpts
plot_concts_bvpts(bvp_cont, channel_for_plot)

### Waveforms: artefact removal

In [ ]:
# for dev
importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
### remove artifact waveforms
from cedalion.sigproc.bvp_wav_ana_v12 import remove_artifact_waveforms
bvp_cont.wav_storage_user, bvp_cont.wav_storage_details = remove_artifact_waveforms(
    bvp_cont['bvp_ts'],
    bvp_cont.wav_storage_user,
    bvp_cont.wav_storage_details
)

In [ ]:
# show updated BVP_Container
print(bvp_cont)

Plot

In [ ]:
# plot waveforms without artefacts
from cedalion.sigproc.bvp_wav_ana_v12 import plot_wavs_woa
plot_wavs_woa(bvp_cont, channel_for_plot)

### Waveforms: classification

In [ ]:
# for dev
importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
### classify waveforms using 'delta' and 'max'
from cedalion.sigproc.bvp_wav_ana_v12 import classify_waveforms
bvp_cont.wav_storage_user, bvp_cont.wav_storage_details = classify_waveforms(
    bvp_cont, 'delta')
bvp_cont.wav_storage_user, bvp_cont.wav_storage_details = classify_waveforms(
    bvp_cont, 'max')

In [ ]:
# show number of deleted waveforms for each channel
for ch in bvp_cont['bvp_ts'].channel.values:
    print(wav_storage_details[ch]["text_num_del_wavs"])

In [ ]:
# show updated BVP_Container
print(bvp_cont)

### @User: choose classification metric for plot

In [ ]:
classfivication_metric = 'delta' # 'max' or 'delta'

In [ ]:
# plot classified waveforms
from cedalion.sigproc.bvp_wav_ana_v12 import plot_wavs_classes
plot_wavs_classes(bvp_cont, channel_for_plot, classfivication_metric)

### BVPA extraction

In [ ]:
# for dev
importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
### extract BVPA
from cedalion.sigproc.bvp_wav_ana_v12 import extract_bvpa
bvpa_ts = extract_bvpa(
    bvp_cont['bvp_ts'].sel(compound="bvp_ts"),
    bvp_cont.wav_storage_user
)
bvp_cont['bvpa_ts'] = bvpa_ts

In [ ]:
# show updated BVP_Container and xarray of BVPA time series
print(bvp_cont)
display(bvp_cont['bvpa_ts'])

Plot

In [ ]:
# plot BVP and BVPA time series
from cedalion.sigproc.bvp_wav_ana_v12 import plot_bvpts_bvpats
plot_bvpts_bvpats(bvp_cont, channel_for_plot)

### Pulse Rate: extraction

In [ ]:
# for dev
importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
### extract pulse rate
from cedalion.sigproc.bvp_wav_ana_v12 import extract_pulse_rate
pulse_rate_ts = extract_pulse_rate(
    bvp_cont['bvp_ts'].sel(compound="bvp_ts"),
    bvp_cont.wav_storage_user
)
bvp_cont['pulse_rate_ts'] = pulse_rate_ts

In [ ]:
# show updated BVP_Container and xarray of pulse rate time series
print(bvp_cont)
display(bvp_cont['pulse_rate_ts'])

Plot

In [ ]:
# plot pulse rate time series
from cedalion.sigproc.bvp_wav_ana_v12 import plot_pulse_rate
plot_pulse_rate(bvp_cont, channel_for_plot)

### Pulse Rate: filtering

In [ ]:
# for dev
importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
### apply band pass filter on pulse rate time series
from cedalion.sigproc.bvp_wav_ana_v12 import filter_pulse_rate
pulse_rate_ts = filter_pulse_rate(
    pulse_rate_ts,
    fmin=0.1, fmax=0.45,
    butter_order=2)
bvp_cont['pulse_rate_ts'] = pulse_rate_ts

In [ ]:
# show updated BVP_Container and xarray of pulse rate time series
print(bvp_cont)
display(bvp_cont['pulse_rate_ts'])

Plot

In [ ]:
# plot BVPA and pulse rate time series and filtered pulse rate time series
from cedalion.sigproc.bvp_wav_ana_v12 import plot_bvpa_pr_comparison
plot_bvpa_pr_comparison(bvp_cont, channel_for_plot)

### comprehensive Plot

In [ ]:
# for dev
importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
# plot [HbO], BVPA and pulse rate time series
from cedalion.sigproc.bvp_wav_ana_v12 import plot_concts_bvpats_pr
plot_concts_bvpats_pr(bvp_cont, channel_for_plot)

### Wavelet Coherence Analysis: BVPA - PR

In [ ]:
# for dev
importlib.reload(cedalion.sigproc.bvp_wav_ana_v12)

In [ ]:
### calculate wavelet coherence between BVPA and pulse rate time series
from cedalion.sigproc.bvp_wav_ana_v12 import calc_wav_coh_bvpa_pr

bvp_cont.wav_storage_details = calc_wav_coh_bvpa_pr(
    bvp_cont['bvpa_ts'].sel(compound='bvpa_smooth'),
    bvp_cont['pulse_rate_ts'].sel(compound='pulse_rate_smooth'),
    bvp_cont.wav_storage_details,
    s0=0.5, J=100)

Plots

In [ ]:
# plot BVPA and pulse rate time series and wavelet coherence plot
from cedalion.sigproc.bvp_wav_ana_v12 import plot_coherence_bvpa_pr
plot_coherence_bvpa_pr(bvp_cont, channel_for_plot)

In [ ]:
# for dev (old plot)
# importlib.reload(cedalion.sigproc.Ablage)
from cedalion.sigproc.Ablage import plot_coherence_bvpa_pr
plot_coherence_bvpa_pr(bvp_cont, channel_for_plot)